In [1]:
# Cell 1 — Config
from pathlib import Path
import pandas as pd, json, re, cv2
from collections import Counter, defaultdict

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MANIFEST = ROOT / "data" / "ocr_frames_4k" / "manifest.csv"
OUT      = ROOT / "data" / "ocr_ws6_4k"
OUT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST)
CAM_WEIGHT = {"back_right": 3, "front": 2, "back_left": 1}

print("crops:", len(manifest))
print("per camera:", manifest.groupby("camera").size().to_dict())
print("per colour:", manifest.groupby("color").size().to_dict())
print("bottles:", manifest.groupby(["video","Sequence"]).size().shape[0])
manifest.head()

crops: 526
per camera: {'back_left': 179, 'back_right': 180, 'front': 167}
per colour: {'black': 444, 'cyan': 82}
bottles: 182


,video,Sequence,t_sec,camera,priority,crop_path,gt_tare,gt_year,color
0,14_43,542436,39.0,front,1,data\ocr_frames_4k\crop_14_43_542436_front.jpg,10.8,2033,black
1,14_43,542436,39.0,back_left,2,data\ocr_frames_4k\crop_14_43_542436_back_left...,10.8,2033,black
2,14_43,542436,39.0,back_right,0,data\ocr_frames_4k\crop_14_43_542436_back_righ...,10.8,2033,black
3,14_43,542437,43.0,front,1,data\ocr_frames_4k\crop_14_43_542437_front.jpg,10.7,2036,black
4,14_43,542437,43.0,back_left,2,data\ocr_frames_4k\crop_14_43_542437_back_left...,10.7,2036,black


In [2]:
# Cell 2 — preprocess (kept mild; metal text dislikes heavy blur)
def preprocess(img_bgr):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(g)
    g = cv2.bilateralFilter(g, 9, 75, 75)
    g = cv2.resize(g, None, fx=3, fy=3, interpolation=cv2.INTER_LANCZOS4)
    return g

In [3]:
import time, cv2, easyocr

def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

reader = easyocr.Reader(['en'], gpu=False, verbose=False)

sample = manifest.sample(8, random_state=0)
for r in sample.itertuples(index=False):
    img = cv2.imread(str(ROOT / r.crop_path))
    h, w = img.shape[:2]

    t0 = time.time()
    reader.readtext(img, detail=0)
    t_orig = time.time() - t0

    img_small = cap_size(img, 640)
    t0 = time.time()
    reader.readtext(img_small, detail=0, canvas_size=736, mag_ratio=1.0)
    t_fast = time.time() - t0

    print(f"{r.camera:10s} {r.Sequence:6d}  size={h}x{w}  orig={t_orig*1000:.0f} ms  fast={t_fast*1000:.0f} ms")

d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\.venv\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(
d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


back_left  542558  size=836x1025  orig=3222 ms  fast=1238 ms
back_right 542584  size=833x1310  orig=3112 ms  fast=829 ms
back_right 542587  size=836x1051  orig=2475 ms  fast=1180 ms
front      542443  size=823x611  orig=2557 ms  fast=1759 ms
back_right     55  size=722x870  orig=2526 ms  fast=1291 ms
back_left  542442  size=673x779  orig=1918 ms  fast=1299 ms
back_right 542565  size=821x1273  orig=4178 ms  fast=998 ms
back_left  542569  size=836x1264  orig=3687 ms  fast=1082 ms


In [4]:
import torch, os, time, cv2, easyocr

# 1. use all CPU cores
torch.set_num_threads(os.cpu_count())
torch.set_num_interop_threads(1)
print("torch threads:", torch.get_num_threads(), "| cpu count:", os.cpu_count())

# 2. helper
def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

# 3. reader
reader = easyocr.Reader(['en'], gpu=False, verbose=False)

# 4. quick benchmark on 5 crops
sample = manifest.sample(5, random_state=0)
for r in sample.itertuples(index=False):
    img = cv2.imread(str(ROOT / r.crop_path))
    img = cap_size(img, 640)
    t0 = time.time()
    reader.readtext(img, detail=0, canvas_size=736, mag_ratio=1.0,
                    batch_size=1, decoder="greedy", paragraph=False)
    print(f"{r.camera:10s} {r.Sequence:>8}  size={img.shape[0]}x{img.shape[1]}  {(time.time()-t0)*1000:.0f} ms")

torch threads: 16 | cpu count: 16
back_left    542558  size=521x640  1436 ms
back_right   542584  size=406x640  1009 ms
back_right   542587  size=509x640  1419 ms
front        542443  size=640x475  1697 ms
back_right       55  size=531x640  1329 ms


In [ ]:
# Cell 3 — cap_size defined first, then OCR loop
import easyocr, time

def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

reader = easyocr.Reader(['en'], gpu=False, verbose=False)

TARE_RE = re.compile(r'(\d{1,2})[,.](\d)\s*[Kk][Gg]')
YEAR_RE = re.compile(r'\b(19|20)\d{2}\b')
STICKER_KG = {12.5, 10.5}

def parse(txts):
    joined = " ".join(txts).replace(" ", "")
    tare = None
    m = TARE_RE.search(joined)
    if m:
        v = float(f"{m.group(1)}.{m.group(2)}")
        if v not in STICKER_KG and 5.0 <= v <= 25.0:
            tare = v
        else:
            for tok in re.findall(r'\d{1,2}[,.]\d', joined):
                vv = float(tok.replace(",", "."))
                if vv not in STICKER_KG and 5.0 <= vv <= 25.0:
                    tare = vv; break
    year = None
    for tok in txts:
        y = YEAR_RE.search(tok)
        if y and 1990 <= int(y.group(0)) <= 2035:
            year = int(y.group(0)); break
    return tare, year

manifest = manifest.drop_duplicates(subset=["video", "Sequence", "camera"]).reset_index(drop=True)
print(f"manifest: {len(manifest)} crops")

crop_rows = []
t_start = time.time()

for i, r in enumerate(manifest.itertuples(index=False), 1):
    img = cv2.imread(str(ROOT / r.crop_path))
    if img is None:
        continue
    img = cap_size(img, 640)
    txts = reader.readtext(
        img, detail=0,
        canvas_size=736,
        mag_ratio=1.0,
        batch_size=8,
        decoder="greedy",
    )
    tare, year = parse(txts)
    crop_rows.append({
        "video": r.video, "Sequence": int(r.Sequence),
        "camera": r.camera, "priority": int(r.priority),
        "color": r.color,
        "gt_tare": r.gt_tare, "gt_year": r.gt_year,
        "raw": str(txts),
        "tare": tare, "year": year,
    })
    if i % 40 == 0:
        elapsed = time.time() - t_start
        per_crop = elapsed / i
        remaining = per_crop * (len(manifest) - i)
        print(f"  {i}/{len(manifest)}  ({elapsed:.0f}s elapsed, ~{remaining:.0f}s left, {per_crop*1000:.0f} ms/crop)")

crop_df = pd.DataFrame(crop_rows)
crop_df.to_csv(OUT / "ocr_per_crop.csv", index=False)
print(f"\ncrops OCR'd: {len(crop_df)}",
      f"| crops with tare: {int(crop_df.tare.notna().sum())}",
      f"| crops with year: {int(crop_df.year.notna().sum())}",
      f"| total time: {time.time()-t_start:.0f}s")


=== 14_43  Sequence 542436  (black, GT: tare=10.8  year=2033) ===
  front       tare=—       year=—     raw=['3', 'Umen', 'Es']
  back_left   tare=—       year=—     raw=['PRIM']
  back_right  tare=—       year=—     raw=[]

=== 14_43  Sequence 542437  (black, GT: tare=10.7  year=2036) ===
  front       tare=—       year=—     raw=[]
  back_left   tare=—       year=—     raw=['8* .', 'Un1965', 'KG-2037 -', '10.5', '64z']
  back_right  tare=—       year=—     raw=['8', '0', '1o _', 'P15Y-T', ')', 'PRORAAN', '615', 'UN1965', 'KG-2037']

=== 14_43  Sequence 542439  (black, GT: tare=10.6  year=2036) ===
  front       tare=—       year=—     raw=[]
  back_left   tare=—       year=—     raw=['PRIMAGI']
  back_right  tare=—       year=—     raw=['PiSv U04955,', '4']

=== 14_43  Sequence 542440  (black, GT: tare=10.6  year=2036) ===
  front       tare=—       year=—     raw=['7', '10.6', '3', 'UN1965', 'Pasv D', 'K6re32', 'Tarea']
  back_left   tare=—       year=—     raw=['~ROPAAN', 'UN1965'

In [6]:
# Cell 4 — weighted majority vote per bottle, per-camera and per-colour metrics
def weighted_vote(rows, field):
    scores = defaultdict(float)
    for r in rows:
        v = r.get(field)
        if v is None or pd.isna(v):
            continue
        scores[v] += CAM_WEIGHT.get(r.get("camera"), 1)
    if not scores:
        return None
    return max(scores.items(), key=lambda kv: kv[1])[0]

def plain_vote(rows, field):
    vals = [r[field] for r in rows if r.get(field) is not None and not pd.isna(r.get(field))]
    return Counter(vals).most_common(1)[0][0] if vals else None

per_bottle = []
for (vk, seq), grp in crop_df.groupby(["video", "Sequence"]):
    rows = grp.to_dict("records")
    per_bottle.append({
        "video": vk, "Sequence": seq,
        "color": rows[0]["color"],
        "n_cameras_attempted": len(rows),
        "cameras": "+".join(sorted(r["camera"] for r in rows)),
        "tare":       weighted_vote(rows, "tare"),
        "year":       weighted_vote(rows, "year"),
        "tare_plain": plain_vote(rows, "tare"),
        "year_plain": plain_vote(rows, "year"),
        "gt_tare":    rows[0]["gt_tare"],
        "gt_year":    rows[0]["gt_year"],
    })

bottle_df = pd.DataFrame(per_bottle)
bottle_df["tare_ok"]       = (bottle_df.tare       == bottle_df.gt_tare).astype(int)
bottle_df["year_ok"]       = (bottle_df.year       == bottle_df.gt_year).astype(int)
bottle_df["tare_ok_plain"] = (bottle_df.tare_plain == bottle_df.gt_tare).astype(int)
bottle_df["year_ok_plain"] = (bottle_df.year_plain == bottle_df.gt_year).astype(int)

# overall
n = len(bottle_df)
overall = {
    "n_bottles":               n,
    "tare_exact_match_weighted": round(int(bottle_df.tare_ok.sum()) / n, 3),
    "year_exact_match_weighted": round(int(bottle_df.year_ok.sum()) / n, 3),
    "both_exact_match_weighted": round(int(((bottle_df.tare_ok == 1) & (bottle_df.year_ok == 1)).sum()) / n, 3),
    "tare_exact_match_plain":    round(int(bottle_df.tare_ok_plain.sum()) / n, 3),
    "year_exact_match_plain":    round(int(bottle_df.year_ok_plain.sum()) / n, 3),
}

# per-colour
per_color = {}
for color, grp in bottle_df.groupby("color"):
    m = len(grp)
    per_color[color] = {
        "bottles": m,
        "tare_exact_match_weighted": round(int(grp.tare_ok.sum()) / m, 3),
        "year_exact_match_weighted": round(int(grp.year_ok.sum()) / m, 3),
        "both_exact_match_weighted": round(int(((grp.tare_ok == 1) & (grp.year_ok == 1)).sum()) / m, 3),
    }

# back-right alone
br = crop_df[crop_df.camera == "back_right"].copy()
br = br.drop_duplicates(subset=["video", "Sequence"]).reset_index(drop=True)

# diagnostics: guard against every type of silent surprise
print("br rows:", len(br))
print("br columns:", list(br.columns))

br_metrics = {
    "back_right_crops":         int(len(br)),
    "back_right_tare_hits":     int(br["tare"].notna().sum()),
    "back_right_year_hits":     int(br["year"].notna().sum()),
    "back_right_tare_correct":  int((br["tare"] == br["gt_tare"]).sum()),
    "back_right_year_correct":  int((br["year"] == br["gt_year"]).sum()),
}
print("br_metrics:", br_metrics)

result = {
    "overall": overall,
    "per_color": per_color,
    "back_right_only": br_metrics,
    "coverage": {
        "gt_bottles_targeted": int(len(manifest.groupby(['video','Sequence']).size())),
        "bottles_with_3_cameras": int((bottle_df.n_cameras_attempted == 3).sum()),
        "bottles_with_2_cameras": int((bottle_df.n_cameras_attempted == 2).sum()),
        "bottles_with_1_camera":  int((bottle_df.n_cameras_attempted == 1).sum()),
        "total_crops":            int(len(crop_df)),
    },
}
print(json.dumps(result, indent=2))
(OUT / "ocr_metrics.json").write_text(json.dumps(result, indent=2))
bottle_df.to_csv(OUT / "ocr_per_bottle.csv", index=False)
print("\nsaved:", (OUT / "ocr_metrics.json").relative_to(ROOT))

br rows: 180
br columns: ['video', 'Sequence', 'camera', 'priority', 'color', 'gt_tare', 'gt_year', 'raw', 'tare', 'year']
br_metrics: {'back_right_crops': 180, 'back_right_tare_hits': 1, 'back_right_year_hits': 7, 'back_right_tare_correct': 1, 'back_right_year_correct': 0}
{
  "overall": {
    "n_bottles": 182,
    "tare_exact_match_weighted": 0.011,
    "year_exact_match_weighted": 0.0,
    "both_exact_match_weighted": 0.0,
    "tare_exact_match_plain": 0.011,
    "year_exact_match_plain": 0.0
  },
  "per_color": {
    "black": {
      "bottles": 154,
      "tare_exact_match_weighted": 0.013,
      "year_exact_match_weighted": 0.0,
      "both_exact_match_weighted": 0.0
    },
    "cyan": {
      "bottles": 28,
      "tare_exact_match_weighted": 0.0,
      "year_exact_match_weighted": 0.0,
      "both_exact_match_weighted": 0.0
    }
  },
  "back_right_only": {
    "back_right_crops": 180,
    "back_right_tare_hits": 1,
    "back_right_year_hits": 7,
    "back_right_tare_correct": 1,

In [12]:
# diagnostic — filter crops by Pr_tare_gram
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

gram_map = {}
for vk, path, sheet in [
    ("14_43", ROOT/"data"/"14_43_ground_truth_ai_app.xlsx", "14_43_ground_truth"),
    ("14_55", ROOT/"data"/"14_55_ground_truth_ai_app.xlsx", "14_55_ground_truth"),
]:
    df = pd.read_excel(path, sheet_name=sheet)
    if "Sequence" not in df.columns:
        df["Sequence"] = range(len(df))

    # ← new: skip rows where Sequence or Pr_tare_gram is missing
    df = df.dropna(subset=["Sequence", "Pr_tare_gram"]).copy()
    df["Sequence"] = df["Sequence"].astype(int)

    for _, r in df.iterrows():
        gram_map[(vk, int(r["Sequence"]))] = float(r["Pr_tare_gram"]) / 1000.0

print("gram_map entries:", len(gram_map))

# attach gram_ref to bottle_df and compute plausibility
bottle_df["gram_ref"] = bottle_df.apply(
    lambda r: gram_map.get((r["video"], int(r["Sequence"])), None), axis=1
)

bottle_df["tare_plausible"] = bottle_df.apply(
    lambda r: pd.notna(r["tare"]) and pd.notna(r["gram_ref"])
              and abs(float(r["tare"]) - float(r["gram_ref"])) <= 0.5,
    axis=1,
)

n_total = len(bottle_df)
n_plausible = int(bottle_df.tare_plausible.sum())

print(f"\nbottles: {n_total}")
print(f"bottles with a plausible tare (|OCR - Pr_tare_gram| <= 0.5 kg): {n_plausible}")
print("\nthe plausible ones:")
if n_plausible > 0:
    print(bottle_df[bottle_df.tare_plausible][
        ["video", "Sequence", "tare", "gram_ref", "gt_tare"]
    ].to_string(index=False))
else:
    print("  (none)")

gram_map entries: 249

bottles: 182
bottles with a plausible tare (|OCR - Pr_tare_gram| <= 0.5 kg): 4

the plausible ones:
video  Sequence  tare  gram_ref  gt_tare
14_43    542440  10.6      10.6     10.6
14_43    542524  10.8      10.6     10.6
14_43    542578  10.9      10.9     10.9
14_43    542593  15.7      15.7     15.7


In [13]:
# look at the 10 bottles closest to the ground truth (most plausible)
bottle_df["gap"] = (bottle_df["tare"] - bottle_df["gram_ref"]).abs()
print(bottle_df.nsmallest(10, "gap")[["video", "Sequence", "tare", "gram_ref", "gt_tare", "gap"]].to_string(index=False))

video  Sequence  tare  gram_ref  gt_tare  gap
14_43    542440  10.6      10.6     10.6  0.0
14_43    542578  10.9      10.9     10.9  0.0
14_43    542593  15.7      15.7     15.7  0.0
14_43    542524  10.8      10.6     10.6  0.2
14_43    542557  15.5      16.1     16.1  0.6
14_43    542540   6.2      11.0     11.0  4.8
14_43    542558  10.8      15.7     15.7  4.9
14_43    542441  10.6      15.6     15.6  5.0
14_43    542574  10.6      15.6     15.6  5.0
14_43    542555  10.7      15.8     15.8  5.1


In [ ]:
# Cell 5 — sanity check
print("crops:", len(crop_df),
      "| with tare:", int(crop_df.tare.notna().sum()),
      "| with year:", int(crop_df.year.notna().sum()))
print("\n5 random raw-text examples:")
for t in crop_df.sample(min(5, len(crop_df)), random_state=0)["raw"].tolist():
    print("  ", t[:140])
print("\n5 random crops where tare WAS parsed:")
parsed = crop_df[crop_df.tare.notna()]
if len(parsed) > 0:
    for _, r in parsed.sample(min(5, len(parsed)), random_state=0).iterrows():
        print(f"  {r.video} seq={r.Sequence} cam={r.camera} tare={r.tare} gt={r.gt_tare}  raw={r.raw[:80]}")
else:
    print("  (none)")

crops: 526 | with tare: 2 | with year: 11

5 random raw-text examples:
   ['KG-2037', 'UN1965 .', 'P1SY-N', '10.8', 'AGaz']
   ['UN 1955', 'PiSY']
   ['Je', '3']
   ['"5', 'Wng', 'PYIMAGAZ', '0@', '70', '200b]ot', '3a', '1', 'Fobs [', 'Tobs', '49']
   ['10),65', 'PRIMAGA ']

5 random crops where tare WAS parsed:
  14_43 seq=542593 cam=back_left tare=15.7 gt=15.7  raw=['02', '15,7 KG', 'ANE', 'PROP', '2']
  14_43 seq=542440 cam=back_right tare=10.6 gt=10.6  raw=['Fport', '6', '10.6', 'KG-2032']


In [10]:
# Diagnostic — no GT matching, just what OCR saw per bottle
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_IN = ROOT / "data" / "ocr_ws6_4k"

# load the raw OCR output that already exists
crop_df = pd.read_csv(OUT_IN / "ocr_per_crop.csv")

# group by (video, Sequence) and print every crop with its raw text + parsed values
for (vk, seq), grp in crop_df.groupby(["video", "Sequence"]):
    gt_tare = grp.iloc[0]["gt_tare"]
    gt_year = grp.iloc[0]["gt_year"]
    colour  = grp.iloc[0]["color"]

    print(f"\n=== {vk}  Sequence {seq}  ({colour}, GT: tare={gt_tare}  year={gt_year}) ===")
    for _, r in grp.iterrows():
        cam   = r["camera"]
        raw   = str(r["raw"])[:140]
        tare  = r["tare"] if pd.notna(r["tare"]) else "—"
        year  = r["year"] if pd.notna(r["year"]) else "—"
        print(f"  {cam:10s}  tare={str(tare):6s}  year={str(year):4s}  raw={raw}")


=== 14_43  Sequence 542436  (black, GT: tare=10.8  year=2033) ===
  front       tare=—       year=—     raw=['3', 'Umen', 'Es']
  back_left   tare=—       year=—     raw=['PRIM']
  back_right  tare=—       year=—     raw=[]

=== 14_43  Sequence 542437  (black, GT: tare=10.7  year=2036) ===
  front       tare=—       year=—     raw=[]
  back_left   tare=—       year=—     raw=['8* .', 'Un1965', 'KG-2037 -', '10.5', '64z']
  back_right  tare=—       year=—     raw=['8', '0', '1o _', 'P15Y-T', ')', 'PRORAAN', '615', 'UN1965', 'KG-2037']

=== 14_43  Sequence 542439  (black, GT: tare=10.6  year=2036) ===
  front       tare=—       year=—     raw=[]
  back_left   tare=—       year=—     raw=['PRIMAGI']
  back_right  tare=—       year=—     raw=['PiSv U04955,', '4']

=== 14_43  Sequence 542440  (black, GT: tare=10.6  year=2036) ===
  front       tare=—       year=—     raw=['7', '10.6', '3', 'UN1965', 'Pasv D', 'K6re32', 'Tarea']
  back_left   tare=—       year=—     raw=['~ROPAAN', 'UN1965'